# Monday March 30th 2026
## Announcements
- Assignment 5 due on April 2nd at 11:59pm
- Final exam Saturday April 11th

# Chapter 14: Symbolic computing: `sympy`

So far, we have focussed on *numerical* computation. Our focus was to give *approximate* numerical values to complex expressions.

*Symbolic computation* deals with the *exact* computation of general mathematical objects. Symbolic computation software are often called Computer Algebra Systems (CAS). Some of the most popular CAS are Mathematica (from Wolfram Research in Champaign Illinois), Maple (from Maple Software in Waterloo), sageMath (connected to U. of Washington), the symbolic toolbox in matlab (formerly mupad from University of Paderborn, Germany), or Macaulay2 (MIT). 

We will focus on [Sympy](https://sympy.org), which has the advantage of using python syntax and interfacing with matplotlib, numpy etc.



## Numeric vs symbolic computation
A first use of symbolic computation would be to perform exact algebra. Consider the following calculation:

Let $x = \sqrt{27}$ and $y = \sqrt{3}$. Then $x/y = \sqrt{27}/\sqrt{3} = 3\sqrt{3} / \sqrt{3} = 3$.


Let's do this computation with python / numpy:

In [80]:
import numpy as np
x = np.sqrt(27)
y = np.sqrt(3)
print (x/y)

3.0000000000000004


Let's repeat using *symbolic computing*:

In [2]:
import sympy as sp
x = sp.sqrt(27)
y = sp.sqrt(3)
x,y,x/y

(3*sqrt(3), sqrt(3), 3)

Loosely speaking, `np.sqrt(3)` represent the number obtained by taking the square root of 3. `sympy.sqrt(3)` represent the sequence of operations consisting in taking a number (3) and its square root. When evaluating `sympy.sqrt(27) / sympy.sqrt(3)`, sympy evaluates the sequence of operations involved. The `/` and `sqrt` operators are complex rules, capable of doing automatic simplifications.
See for instance:

In [82]:
sp.sqrt(27)

3*sqrt(3)

## 14.1 Some examples of what sympy can do:


In [84]:
x = sp.Symbol('x')
sp.diff(sp.sin(x)*sp.exp(x), x, x)

2*exp(x)*cos(x)

In [85]:
sp.integrate(x**2*sp.sin(x), x)

-x**2*cos(x) + 2*x*sin(x) + 2*cos(x)

In [ ]:
a = sp.Symbol('a', positive=True)
# Notice: We need to make an assumption of positivity in order to 
# find a nice form for the antiderivative.
sp.integrate(sp.sqrt(a**2-x**2)/x**2, x,a)

-asin(x/a) - sqrt(a**2 - x**2)/x

In [87]:
sp.limit(sp.sin(x)/x, x, 0)

1

In [ ]:
sp.solve(x**2 - a, x)

[-sqrt(a), sqrt(a)]


In [90]:
sp.Matrix([[x, 2], [y, 2]])**2

Matrix([
[     x**2 + 2*sqrt(3),       2*x + 4],
[sqrt(3)*x + 2*sqrt(3), 2*sqrt(3) + 4]])

In [92]:
A = sp.Matrix([[1, 2, 2, 4, 6],[1, 2, 3, 6, 9],[1,2,4,8,12]])
A

Matrix([
[1, 2, 2, 4,  6],
[1, 2, 3, 6,  9],
[1, 2, 4, 8, 12]])

In [ ]:
A = sp.Matrix([[1, 2, 2, 4, 6],[1, 2, 3, 6, 9],[1,2,4,8,12]])
Arref, pivots = A.rref()
# reduced row echelon form of the matrix A
Arref


Matrix([
[1, 2, 0, 0, 0],
[0, 0, 1, 2, 3],
[0, 0, 0, 0, 0]])

In [ ]:
# which columns (by index) have pivot positions
pivots

(0, 2)

In [94]:
A.nullspace()

[Matrix([
 [-2],
 [ 1],
 [ 0],
 [ 0],
 [ 0]]),
 Matrix([
 [ 0],
 [ 0],
 [-2],
 [ 1],
 [ 0]]),
 Matrix([
 [ 0],
 [ 0],
 [-3],
 [ 0],
 [ 1]])]

In [95]:
A.rank()

2

In [96]:
a = sp.Symbol('a')
M = sp.Matrix([[a,1,1],[1,a,1],[1,1,2]])
d = M.det()
d

2*a**2 - 2*a

In [97]:
M.inv()

Matrix([
[(2*a - 1)/(2*a**2 - 2*a),        -1/(2*a**2 - 2*a),      -1/(2*a)],
[       -1/(2*a**2 - 2*a), (2*a - 1)/(2*a**2 - 2*a),      -1/(2*a)],
[                -1/(2*a),                 -1/(2*a), (a + 1)/(2*a)]])

## 14.2 Numeral types
python and numpy have integers, complex and floats. The latter are note condusive to symbolic computing (remember how floats are only *approximations* of reals).

`sympy` introduces its own type of numbers: `sp.Integer`, `sp.rational`, `sp.Complex` and `sp.Real`.

In [98]:
a = sp.Integer(2)
b = sp.Integer(3)
print('a: ', type(a), a)
print('b: ', type(b), b)
print('a/b: ', type(a/b), a/b)

a:  <class 'sympy.core.numbers.Integer'> 2
b:  <class 'sympy.core.numbers.Integer'> 3
a/b:  <class 'sympy.core.numbers.Rational'> 2/3


Be careful when creating rational numbers:

In [99]:
print(sp.srepr(22/7))
print(sp.srepr(sp.Integer(22) / sp.Integer(7)))
print(sp.srepr(sp.Rational(22,7)))

3.142857142857143
Rational(22, 7)
Rational(22, 7)


Why does this happen?

**Hint:** Think back to the first week ... how does Python deal with expressions?

## 14.3 `Symbols`
Just sympy is capable of representing the square root in `sqrt(27)` as "the operation of taking the square root" of something that happens to be 27, it can deal with *symbols*.

sympy `Symbol` are just like mathematical symbols. They can be manipulated to form expressions

In [100]:
x = sp.Symbol('x')
y = sp.Symbol('y')
expr = x + 2*y
expr

x + 2*y

In [101]:
(expr - x)**2

4*y**2

In [102]:
x * expr

x*(x + 2*y)

For the sake of it, we can peek at the representation of these expressions:

In [103]:
sp.srepr(x*(x + 2*y))

"Mul(Symbol('x'), Add(Symbol('x'), Mul(Integer(2), Symbol('y'))))"

Note that the statement `x = sp.Symbol('x')` creates a symbol which will be printed as 'x' and assigns it to the python variable `x`. 
Technically, there is nothing wrong with the following. Just don't do it...

In [104]:
r = sp.Symbol('theta')
theta = sp.Symbol('r')
u = r * sp.cos(theta)
u

theta*cos(r)

# Tuesday March 31st 2026

## 14.4 Basic operations on sympy expressions

### Substitution

`subs` can be used to do mathematical substitutions. Note that it does *not* change the expression (sympy expressions are immutable) on which it acts

In [105]:
mexpr = sp.Matrix([2*x+y, 3*x-y])
s,t = sp.symbols(('s','t'))
# Thus is a shortcut for
# s = Symbol('s')
# t = Symbol('t')
# Note that sp.Symbol is a type, sp.symbols is a function
mexpr.subs([(x, s+t),(y,s-t)])

Matrix([
[  3*s + t],
[2*s + 4*t]])

in passing, this is composing x = s+t and y = s-t into (2*x+y, 3*x-y), which is how I introduced matrices... And by the way: 

In [106]:
np.array([[2,1],[3,-1]])@np.array([[1,1],[1,-1]])

array([[3, 1],
       [2, 4]])

In [107]:
a = sp.Symbol('a', positive=True)
x = sp.Symbol('x')
trig_expr = sp.integrate(sp.sqrt(a**2-x**2)/x**2, x)
trig_expr

-asin(x/a) - sqrt(a**2 - x**2)/x

In [ ]:
trig_expr.subs(a,1)

### Manipulating expressions: `expand`, `factor`, `collect`, and `simplify`

Students often ask if results need to be "simplified". This is hard to answer, because the meaning of "simplify" is ambiguous. 

sympy has a series of functions that manipulate expressions:

#### `expand`

In [3]:
x, y = sp.symbols(('x', 'y'))
sp.expand((x + 1)*(x - 2) - 2*(x - 1)*y)

x**2 - 2*x*y - x + 2*y - 2

In [4]:
sp.expand(sp.sin(x+y))

sin(x + y)

In [5]:
sp.expand_trig(sp.sin(x+y))

sin(x)*cos(y) + sin(y)*cos(x)

#### `factor`

In [6]:
#factor will only automatically factor rational roots
x = sp.Symbol('x')
sp.factor(x**3 - x**2 + x - 1)

(x - 1)*(x**2 + 1)

What goes wrong here?

In [9]:
x = sp.Symbol('x')
sp.factor(x**3 - 7*x**2/2 + 4*x - 3/2)

1.5*(0.666666666666667*x - 1.0)*(1.0*x - 1.0)**2

In [10]:
sp.factor(x**3 - 7*x**2/2 + 4*x - sp.Integer(3)/sp.Integer(2))

(x - 1)**2*(2*x - 3)/2

Note in passing that sympy accepts a string describing an expression in most functions that take an expression as argument, so we could also have written

In [ ]:
x = sp.Symbol('x')
sp.factor('x**3 - 7*x**2/2 + 4*x - 3/2')

in which case, it is smart enough to understand that 3/2 denoted a rational number...

#### `collect`

In [11]:
x,y,z = sp.symbols(['x','y','z'])
expr = x*y + x - 3 + 2*x**2 - z*x**2 + x**3*z

c_expr = sp.collect(expr, x)
c_expr

x**3*z + x**2*(2 - z) + x*(y + 1) - 3

In [12]:
zexpr = sp.collect(expr,z)
zexpr

2*x**2 + x*y + x + z*(x**3 - x**2) - 3

#### `simplify`
Some very basic simplifications are automatic ( adding and subtracting the same expression, for instance). In other cases you need to manually instruct sympy to simplify an expression (this is because simplify can be costly).

In [15]:
x = sp.Symbol('x')
expr = x + sp.sin(x) - x/x

expr

x + sin(x) - 1

In [16]:
x = sp.Symbol('x')
expr = sp.sin(x)**2 + x + sp.cos(x)**2
expr

x + sin(x)**2 + cos(x)**2

By itself, sympy did not simplify $\sin(x)^2 + \cos(x)^2 = 1$, but it will if instructed to simplify this expression

In [17]:
expr2 = sp.simplify(expr)
expr2

x + 1

In [18]:
x = sp.Symbol('x')
sp.simplify(sp.sqrt(x**2))

sqrt(x**2)

Why isn't sqrt(x^2) simplified?

In the complex plane, the square root function is bi-valued (for instance, $i^2 = (-1)^2 = -1$), so the simplification $\sqrt{x^2} = |x|$ is only valid for reals. We can tell sympy that $x$ is a real:

In [19]:
x = sp.Symbol('x',real=True)
sp.simplify(sp.sqrt(x**2))


Abs(x)

In [20]:
x = sp.Symbol('x',negative=True)
sp.simplify(sp.sqrt(x**2))

-x

### Example - Sympy is not magic

Consider the polynomial $x^2+2x+1$ and observe how Sympy simplifies this expression:

In [21]:
expr = x**2 + 2*x +1 
expr

x**2 + 2*x + 1

In [22]:
expr.simplify()

x**2 + 2*x + 1

If we wanted instead to have $(x+1)^2$, we need to help Sympy out.

In [23]:
expr.factor()

(x + 1)**2

Other assumptions include
`finite`, `infinite`, `real`, `imaginary`, `rational`, `irrational`, `integer`, `even`, `odd`,`noninteger`, `zero`, `nonzero`, `positive`, `negative`, `nonpositive`, `nonnegative`, `extended_positive`, etc

In [ ]:
n = sp.Symbol('n', integer = True, even = True)
sp.simplify((-1)**n + sp.cos(n*sp.pi))

### Numerical evaluation
Sometimes, we may want to get the numerical value of a complex expression. we can use `evalf` to do this:

In [ ]:
sp.pi.evalf(100)

In [ ]:
sp.exp(1).evalf(32)

In [ ]:
x = sp.Symbol('x')
a = sp.Symbol('a', positive=True)
trig_expr = sp.integrate(sp.sqrt(a**2-x**2)/x**2, x)
expr4 = trig_expr.subs(a,1)
expr4

In [ ]:
trig_expr.subs([(x,1), (a,1)])

In [ ]:
trig_expr.subs([(x,1),(a,1)]).evalf()

### "Lambdification"

A `sympy` expression contains all the informations required to turn symbols into a result, *i.e.* conceptually they can be seen as python functions.

As a matter of fact, they can be turned into python functions:


In [ ]:
expr4

In [ ]:
f = sp.lambdify(x, expr4)
type(f)

In [ ]:
f(1)

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots()
X = np.linspace(-0.99,0.99,200)
ax.plot(X,f(X), label = f'${sp.latex(expr4)}$')
ax.spines[['left', 'bottom']].set_position('zero')
ax.spines[['top', 'right']].set_visible(False)
ax.legend()
plt.show()

In [ ]:
plt.close('all')
fig, ax = plt.subplots()
for aa in (1, 2, 5): 
    expra = trig_expr.subs(a,aa)
    print(expra)
    f = sp.lambdify(x,expra)
    X = np.linspace(-aa, aa, 200)
    ax.plot(X,f(X), label = f'${sp.latex(expra)}$')
ax.spines[['left', 'bottom']].set_position('zero')
ax.spines[['top', 'right']].set_visible(False)
ax.set_xlim([-1,1])
ax.set_ylim([-20,20])
ax.legend()
plt.show()